In [1]:
!pip uninstall -y torchvision
!pip install -U transformers accelerate soundfile librosa pydub

Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 115.2 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.12.0
    Uninstalling transformers-5.12.0:
      Successfully uninstalled transformers-5.12.0


In [12]:
AUDIO_WAV = "/content/drive/MyDrive/мага/МД/narty_16k.wav"
CLEAN_TEXT_FILE = "/content/drive/MyDrive/мага/МД/clean_ossetian.txt"

In [13]:
import os

print("Аудио найдено:", os.path.exists(AUDIO_WAV))
print("Текст найден:", os.path.exists(CLEAN_TEXT_FILE))

Аудио найдено: True
Текст найден: True


In [14]:
with open(CLEAN_TEXT_FILE, "r", encoding="utf-8") as f:
    clean_lines = [line.strip() for line in f if line.strip()]

print("Количество предложений:", len(clean_lines))
print(clean_lines[:5])

Количество предложений: 38
['\ufeffНартæн сæ цæхæрадоны задис иу фæткъуы бæлас; йæ дидинджытæ-иу æрттывтой æрвыгау, æмæ йыл задис иунæг фæткъуы.', 'Фæткъуы уыдис сызгъæрин фæткъуы, зынгау æрттывдтытæ калдта.', 'Æмæ уыдис æлутоны хос адæмæн, иу адзал не здæхта фæстæмæ, уыййедтæмæ цы хъæдгом нæ дзæбæх кодта, цы низæй нæ ирвæзын кодта, ахæм нæ уыдис.', 'Бон-изæрмæ-иу арæгъæд ис уыцы фæткъуы, æхсæв та-иу æй цыдæр адавта.', 'Æмæ йæ хъахъхъæдтой радыгай Нарт; æмæ йæ ничи фæрæзта бахъахъхъæнын.']


In [15]:
import re

with open(CLEAN_TEXT_FILE, "r", encoding="utf-8-sig") as f:
    full_text = f.read()

# Убираем лишние переносы строк и пробелы
full_text = re.sub(r"\s+", " ", full_text).strip()

# Делим текст на предложения по точке, вопросительному, восклицательному знаку и многоточию
clean_lines = re.findall(r"[^.!?…]+[.!?…]?", full_text)

# Чистим пробелы
clean_lines = [s.strip() for s in clean_lines if s.strip()]

print("Количество предложений:", len(clean_lines))
print(clean_lines[:10])
print(clean_lines[-5:])

Количество предложений: 33
['Нартæн сæ цæхæрадоны задис иу фæткъуы бæлас; йæ дидинджытæ-иу æрттывтой æрвыгау, æмæ йыл задис иунæг фæткъуы.', 'Фæткъуы уыдис сызгъæрин фæткъуы, зынгау æрттывдтытæ калдта.', 'Æмæ уыдис æлутоны хос адæмæн, иу адзал не здæхта фæстæмæ, уыййедтæмæ цы хъæдгом нæ дзæбæх кодта, цы низæй нæ ирвæзын кодта, ахæм нæ уыдис.', 'Бон-изæрмæ-иу арæгъæд ис уыцы фæткъуы, æхсæв та-иу æй цыдæр адавта.', 'Æмæ йæ хъахъхъæдтой радыгай Нарт; æмæ йæ ничи фæрæзта бахъахъхъæнын.', 'Уæд иу бон æрзылдис Уæрхæгæн йæхи рад.', 'Æрбасидтис йæ фырттæм, Æхсар æмæ Æхсæртæгмæ, Уæрхæг æмæ сын загъта: - Ай уын фæндæггаг.', 'Ацæут, мæ хуртæ, æмæ уæ цæхæрадон бахъахъхъæнут.', 'Кæннод райсом Æртæ Нарты хæдзарæн лæгæй æрбацæудзысты æмæ уæ иуæн йæ сæр ракæндзысты, иннæмæн – йæ цонг æмæ сæ дыууæ михыл æрсадздзысты, æмæ Æртæ Нарты ‘хсæн дзæгъæлæй баззайдзынæн, æнæ дарæгæй.', '- Ма тæрс, нæ фыд, мах ацæудзыстæм æмæ бæлас бахъахъхъæндзыстæм!']
['Æхсар ын загъта: - Æз дæр цæуын, ды кæдæм цæуай, уырдæм.',

In [16]:
import torch
from transformers import AutoProcessor, Wav2Vec2ForCTC, pipeline

MODEL_ID = "facebook/mms-1b-all"
LANG = "oss"  # осетинский язык

device = "cuda" if torch.cuda.is_available() else "cpu"
pipe_device = 0 if torch.cuda.is_available() else -1

print("Устройство:", device)

print("Загружаю processor...")
processor = AutoProcessor.from_pretrained(MODEL_ID, target_lang=LANG)

print("Загружаю модель...")
model = Wav2Vec2ForCTC.from_pretrained(
    MODEL_ID,
    target_lang=LANG,
    ignore_mismatched_sizes=True,
    low_cpu_mem_usage=True
)

model.to(device)

asr = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    device=pipe_device
)

print("Готово, модель загружена.")

Устройство: cuda
Загружаю processor...
Загружаю модель...


Loading weights:   0%|          | 0/1096 [00:00<?, ?it/s]

Готово, модель загружена.


In [17]:
AUDIO_TEST = "test_3min.wav"

!ffmpeg -y -i "$AUDIO_WAV" -t 00:03:00 -ac 1 -ar 16000 "$AUDIO_TEST"

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [18]:
import os

print(os.path.exists(AUDIO_TEST))

True


In [19]:
print("Распознаю тестовый фрагмент...")

result = asr(
    AUDIO_WAV,
    return_timestamps="word",
    chunk_length_s=20,
    stride_length_s=(2, 2)
)

print("Распознанный текст:")
print(result["text"])

print("\nПервые элементы с таймкодами:")
for chunk in result["chunks"][:20]:
    print(chunk)

Распознаю тестовый фрагмент...
Распознанный текст:
нарты фӕдкъуы нартӕн сӕ цӕхӕрадоны задис иу фӕдкъуы бӕлас йӕ дидинджытӕ-иу ӕрдтывтой ӕрвыгау ӕмӕ йыл задис иунӕг фӕдкъуы фӕдкъуы уыдис сыгъзӕрин фӕдкъуы зынгауӕрдтыфтытӕ калдта ӕмӕ уыдис ӕлутоны хос адӕмӕн иу адзал не здӕхта фӕстӕмӕ уый йӕттӕмӕ цы хъӕдгом нӕ дзӕбӕх кодта цы низӕй нӕй ӕрвӕзын кодта ахӕм нӕуыдис бон изӕрмӕ-иу арӕгъӕд ис уыцы фӕдкъуы ӕхсӕф та-иуӕй цыдӕр адафта ӕмӕ йӕ хъахъхъӕдтой радыгай нарт ӕмӕ йӕ ничи фӕрӕста бахъахъхъӕнын уӕд-иу бон ӕрзылдис уӕрхӕгӕн йӕхи рат ӕрбасидтис йӕ фырттын ӕхсар ӕмӕ ӕхсӕртӕгмӕ уӕрхӕг ӕмӕ сын загъта айуын фӕндаккаг ацӕут мӕгъуртӕ ӕмӕ уӕ цӕгъӕр адон бахъахъхъӕнут кӕннод райсом ӕртӕнарты хӕдзарӕн лӕгӕй ӕрбацӕудзысты ӕмӕ уӕ иуӕн йӕ сӕр ракӕндзысты иннӕмӕн йӕ цонг ӕмӕ сӕ дыууӕ михӕл ӕрсадздзысты ӕмӕ ӕртӕнарты ӕхсӕндзӕгъӕлай баззайдзынӕн ӕнӕдарӕгӕй цӕгъӕрадон уыд ис саджы сыкъатӕй бӕрзондӕхгӕд маӕргъӕр патӕхӕн дӕрӕм нӕ уыд лӕппутӕ загътой матӕр снӕ фыд мах ацӕудзыстӕм ӕмӕ бӕлас бахъахъхъӕндзыстӕм уӕ

In [20]:
MAX_WORD_PAUSE = 0.7

word_chunks = []

for ch in result["chunks"]:
    text = ch.get("text", "").strip()
    timestamp = ch.get("timestamp")

    if not text or timestamp is None:
        continue

    start, end = timestamp

    if start is None or end is None:
        continue

    word_chunks.append({
        "text": text,
        "start": float(start),
        "end": float(end)
    })

print("Количество слов с таймкодами:", len(word_chunks))
print(word_chunks[:10])

Количество слов с таймкодами: 432
[{'text': 'нарты', 'start': 1.28, 'end': 1.7}, {'text': 'фӕдкъуы', 'start': 1.8, 'end': 2.16}, {'text': 'нартӕн', 'start': 4.2, 'end': 4.76}, {'text': 'сӕ', 'start': 5.22, 'end': 5.32}, {'text': 'цӕхӕрадоны', 'start': 5.4, 'end': 6.14}, {'text': 'задис', 'start': 6.24, 'end': 6.66}, {'text': 'иу', 'start': 6.8, 'end': 6.9}, {'text': 'фӕдкъуы', 'start': 6.98, 'end': 7.28}, {'text': 'бӕлас', 'start': 7.32, 'end': 7.68}, {'text': 'йӕ', 'start': 9.26, 'end': 9.36}]


In [21]:
phrases = []

if word_chunks:
    current_words = [word_chunks[0]["text"]]
    current_start = word_chunks[0]["start"]
    current_end = word_chunks[0]["end"]

    for word in word_chunks[1:]:
        pause = word["start"] - current_end

        if pause <= MAX_WORD_PAUSE:
            current_words.append(word["text"])
            current_end = word["end"]
        else:
            phrases.append({
                "text": " ".join(current_words),
                "start": current_start,
                "end": current_end
            })

            current_words = [word["text"]]
            current_start = word["start"]
            current_end = word["end"]

    phrases.append({
        "text": " ".join(current_words),
        "start": current_start,
        "end": current_end
    })

print("Количество фраз:", len(phrases))

for p in phrases[:20]:
    print(f'{p["text"]} | {p["start"]} | {p["end"]}')

Количество фраз: 62
нарты фӕдкъуы | 1.28 | 2.16
нартӕн сӕ цӕхӕрадоны задис иу фӕдкъуы бӕлас | 4.2 | 7.68
йӕ дидинджытӕ-иу ӕрдтывтой ӕрвыгау ӕмӕ йыл задис иунӕг фӕдкъуы | 9.26 | 14.74
фӕдкъуы уыдис сыгъзӕрин фӕдкъуы зынгауӕрдтыфтытӕ калдта | 16.14 | 20.42
ӕмӕ уыдис ӕлутоны хос адӕмӕн | 21.86 | 24.1
иу адзал не здӕхта фӕстӕмӕ | 25.04 | 27.08
уый йӕттӕмӕ | 27.8 | 28.52
цы хъӕдгом нӕ дзӕбӕх кодта цы низӕй нӕй ӕрвӕзын кодта ахӕм нӕуыдис | 29.26 | 34.5
бон изӕрмӕ-иу арӕгъӕд ис уыцы фӕдкъуы | 35.78 | 38.94
ӕхсӕф та-иуӕй цыдӕр адафта | 39.8 | 41.84
ӕмӕ йӕ хъахъхъӕдтой радыгай нарт | 43.28 | 45.4
ӕмӕ йӕ ничи фӕрӕста бахъахъхъӕнын | 46.36 | 49.52
уӕд-иу бон ӕрзылдис уӕрхӕгӕн йӕхи рат | 51.14 | 53.98
ӕрбасидтис йӕ фырттын ӕхсар ӕмӕ ӕхсӕртӕгмӕ уӕрхӕг ӕмӕ сын загъта | 55.2 | 60.62
айуын фӕндаккаг | 61.8 | 62.92
ацӕут мӕгъуртӕ ӕмӕ уӕ цӕгъӕр адон бахъахъхъӕнут | 63.88 | 67.28
кӕннод райсом ӕртӕнарты хӕдзарӕн лӕгӕй ӕрбацӕудзысты | 68.06 | 72.46
ӕмӕ уӕ иуӕн йӕ сӕр ракӕндзысты иннӕмӕн йӕ цонг ӕмӕ сӕ дыу

In [22]:
from difflib import SequenceMatcher
import re
import csv
import os

def normalize_text(text):
    """
    Приводим текст к более простому виду:
    маленькие буквы, без лишних знаков препинания и пробелов.
    Это нужно, чтобы сравнение было мягче.
    """
    text = text.lower()
    text = text.replace("æ", "ӕ").replace("Æ", "ӕ")
    text = re.sub(r"[^\w\sӕӔ]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def similarity(a, b):
    """
    Считаем, насколько похожи две строки.
    1.0 — полностью похожи.
    0.0 — совсем не похожи.
    """
    return SequenceMatcher(None, normalize_text(a), normalize_text(b)).ratio()


THRESHOLD = 0.45   # порог похожести. Если плохо сопоставляет, можно снизить до 0.35
LOOKAHEAD = 10     # сколько следующих распознанных фраз смотрим вперёд
MAX_MERGE = 4      # сколько соседних фраз MMS можно склеивать в одну

aligned = []
j_start = 0

for i, clean_text in enumerate(clean_lines, start=1):
    best = None

    # Ищем подходящую распознанную фразу рядом с текущей позицией
    for j in range(j_start, min(len(phrases), j_start + LOOKAHEAD)):
        combined_text = ""
        start_time = phrases[j]["start"]
        end_time = phrases[j]["end"]

        # Иногда одно предложение может быть разбито на несколько фраз,
        # поэтому пробуем склеить несколько соседних распознанных фрагментов
        for k in range(j, min(len(phrases), j + MAX_MERGE)):
            combined_text = (combined_text + " " + phrases[k]["text"]).strip()
            end_time = phrases[k]["end"]

            score = similarity(clean_text, combined_text)

            if best is None or score > best["score"]:
                best = {
                    "clean_id": i,
                    "clean_text": clean_text,
                    "recognized_text": combined_text,
                    "start": start_time,
                    "end": end_time,
                    "score": score,
                    "j_from": j,
                    "j_to": k
                }

    # Если похожесть достаточная, считаем, что нашли соответствие
    if best and best["score"] >= THRESHOLD:
        aligned.append(best)
        j_start = best["j_to"] + 1
    else:
        # Если ничего нормального не нашли, всё равно сохраняем строку,
        # но без таймкодов
        aligned.append({
            "clean_id": i,
            "clean_text": clean_text,
            "recognized_text": "",
            "start": "",
            "end": "",
            "score": 0,
            "j_from": "",
            "j_to": ""
        })

print("Всего предложений в чистом тексте:", len(clean_lines))
print("Всего сопоставленных строк:", len(aligned))

success = sum(1 for row in aligned if row["start"] != "")
print("Успешно сопоставлено:", success)

print("\nПервые 10 сопоставлений:")
for row in aligned:
    print("-----")
    print("Чистый текст:", row["clean_text"])
    print("Распознано:", row["recognized_text"])
    print("Время:", row["start"], "-", row["end"])
    print("Похожесть:", round(row["score"], 3))

Всего предложений в чистом тексте: 33
Всего сопоставленных строк: 33
Успешно сопоставлено: 31

Первые 10 сопоставлений:
-----
Чистый текст: Нартæн сæ цæхæрадоны задис иу фæткъуы бæлас; йæ дидинджытæ-иу æрттывтой æрвыгау, æмæ йыл задис иунæг фæткъуы.
Распознано: нартӕн сӕ цӕхӕрадоны задис иу фӕдкъуы бӕлас йӕ дидинджытӕ-иу ӕрдтывтой ӕрвыгау ӕмӕ йыл задис иунӕг фӕдкъуы
Время: 4.2 - 14.74
Похожесть: 0.972
-----
Чистый текст: Фæткъуы уыдис сызгъæрин фæткъуы, зынгау æрттывдтытæ калдта.
Распознано: фӕдкъуы уыдис сыгъзӕрин фӕдкъуы зынгауӕрдтыфтытӕ калдта
Время: 16.14 - 20.42
Похожесть: 0.893
-----
Чистый текст: Æмæ уыдис æлутоны хос адæмæн, иу адзал не здæхта фæстæмæ, уыййедтæмæ цы хъæдгом нæ дзæбæх кодта, цы низæй нæ ирвæзын кодта, ахæм нæ уыдис.
Распознано: ӕмӕ уыдис ӕлутоны хос адӕмӕн иу адзал не здӕхта фӕстӕмӕ уый йӕттӕмӕ цы хъӕдгом нӕ дзӕбӕх кодта цы низӕй нӕй ӕрвӕзын кодта ахӕм нӕуыдис
Время: 21.86 - 34.5
Похожесть: 0.966
-----
Чистый текст: Бон-изæрмæ-иу арæгъæд ис уыцы фæткъуы, æхсæв т

In [24]:
import os
import csv
import shutil
from pydub import AudioSegment

# Папка, куда сохраняем итоговый датасет на Google Диске
DATASET_DIR = "/content/drive/MyDrive/ossetian_dataset_final"
WAVS_DIR = os.path.join(DATASET_DIR, "wavs")

# Создаём папки
os.makedirs(DATASET_DIR, exist_ok=True)
os.makedirs(WAVS_DIR, exist_ok=True)

# Загружаем полный аудиофайл
audio = AudioSegment.from_wav(AUDIO_WAV)

# Файлы с метаданными
METADATA_SIMPLE = os.path.join(DATASET_DIR, "metadata.csv")
METADATA_FULL = os.path.join(DATASET_DIR, "metadata_full.csv")

saved = 0

with open(METADATA_SIMPLE, "w", encoding="utf-8", newline="") as simple_file, \
     open(METADATA_FULL, "w", encoding="utf-8", newline="") as full_file:

    simple_writer = csv.writer(simple_file, delimiter="|")
    full_writer = csv.writer(full_file)

    # Полная таблица с дополнительной информацией
    full_writer.writerow([
        "id",
        "audio_path",
        "clean_text",
        "recognized_text",
        "start_sec",
        "end_sec",
        "duration_sec",
        "similarity"
    ])

    for row in aligned:
        # Пропускаем строки, где не удалось найти таймкоды
        if row["start"] == "":
            continue

        clip_id = f'clip_{row["clean_id"]:06d}'
        filename = f"{clip_id}.wav"
        output_path = os.path.join(WAVS_DIR, filename)

        START_PADDING_MS = 150   # небольшой запас перед фразой
        END_PADDING_MS = 500     # запас после фразы, чтобы не обрезать конец

        start_sec = int(float(row["start"]) * 1000)
        end_sec = int(float(row["end"]) * 1000)

        # Добавляем запас, но не выходим за границы аудио
        start_ms = max(0, start_sec - START_PADDING_MS)
        end_sec = min(len(audio), end_sec + END_PADDING_MS)

        if end_sec <= start_sec:
            continue

        # Вырезаем фрагмент из полного аудио
        chunk = audio[start_ms:end_sec]
        chunk.export(output_path, format="wav")

        duration_sec = round(end_sec - start_sec, 3)

        clean_text = row["clean_text"]
        recognized_text = row["recognized_text"]
        similarity = round(row["score"], 3)

        # Простой формат датасета:
        # имя_файла | правильный текст | правильный текст
        # Такой формат часто используют для TTS
        simple_writer.writerow([
            f"wavs/{filename}",
            clean_text,
            clean_text
        ])

        # Полный формат для проверки качества
        full_writer.writerow([
            clip_id,
            f"wavs/{filename}",
            clean_text,
            recognized_text,
            start_sec,
            end_sec,
            duration_sec,
            similarity
        ])

        saved += 1

print("Готово.")
print("Сохранено фрагментов:", saved)
print("Папка датасета:", DATASET_DIR)
print("Простой metadata:", METADATA_SIMPLE)
print("Полный metadata:", METADATA_FULL)

Готово.
Сохранено фрагментов: 31
Папка датасета: /content/drive/MyDrive/ossetian_dataset_final
Простой metadata: /content/drive/MyDrive/ossetian_dataset_final/metadata.csv
Полный metadata: /content/drive/MyDrive/ossetian_dataset_final/metadata_full.csv


In [59]:
import json
from google.colab import files

NOTEBOOK_PATH = "split_by_pauses.ipynb"  # название твоего ноутбука
CLEAN_NOTEBOOK_PATH = "split_by_pauses_clean.ipynb"

with open(NOTEBOOK_PATH, "r", encoding="utf-8") as f:
    nb = json.load(f)

# Удаляем проблемные metadata widgets
if "metadata" in nb:
    nb["metadata"].pop("widgets", None)

# Удаляем outputs и execution_count
for cell in nb.get("cells", []):
    if "metadata" in cell:
        cell["metadata"].pop("widgets", None)

    if cell.get("cell_type") == "code":
        cell["outputs"] = []
        cell["execution_count"] = None

with open(CLEAN_NOTEBOOK_PATH, "w", encoding="utf-8") as f:
    json.dump(nb, f, ensure_ascii=False, indent=1)

print("Готово:", CLEAN_NOTEBOOK_PATH)

files.download(CLEAN_NOTEBOOK_PATH)

Готово: split_by_pauses_clean.ipynb


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>